# Phase 7 — Robustness checks

Added after the group's review of the report. Questions addressed:
1. **Stop-buying model** fully specified (label, features, data, hyperparameters, AUC, Brier score, calibration).
2. **Do the K-Means states help the stop-buying model?** The same XGBoost with and without the state features (Segment_t, Segment_t-1, Segment_t-2, StateDuration).
3. **Does the shorter S7→S8 interval (~2.3 months) change the Phase 6 conclusion?** Computed separately for origin S6 (a full 3-month period) and origin S7.
4. **Markov-order G-test for each time interval** (instead of pooling the training origins) to check that the conclusion is not an artefact of pooling over time.

All definitions, data and parameters are the same as in notebook 06; the test set is used for evaluation only.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier
from config import PROCESSED, INTERIM, TABLES, N_STATES, STATE_NAMES, CURRENT_FEATURES, RANDOM_STATE, TARGET
from decision import (period_revenue, attach_revenue, state_values, downgrade_mask, downgrade_risk, segment_risk,
                      captured_share, bootstrap_captured_diff)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}"); pd.set_option("display.width", 200)

rev = period_revenue(pd.read_parquet(INTERIM / "transactions_clean.parquet"))
ds = attach_revenue(pd.read_parquet(PROCESSED / "prediction_dataset.parquet"), rev)
V = state_values(ds[ds.split == "train"], N_STATES); D = downgrade_mask(V); segr = segment_risk(ds[ds.split == "train"], D, N_STATES)
te = attach_revenue(pd.read_parquet(PROCESSED / "phase4_test_predictions.parquet"), rev)
seg, nxt = te["Segment_t"].astype(int).values, te["Segment_t+1"].astype(int).values
te["down"] = D[seg, nxt]; te["stop"] = (te.v > 0) & (te.rev_next == 0)
te["L_stop"] = np.where(te.stop, te.v, 0.0)
te["r_raw"] = downgrade_risk(te[[f"proba_XGBoost_B_{k}" for k in range(N_STATES)]].values, seg, D)
te["EL_state"] = te.r_raw * te.v; te["SEG_EL"] = segr.reindex(seg).values * te.v

## 1. Stop-buying model: full specification, with and without the K-Means state features

In [2]:
dh = ds[ds["history_eligible"]].copy(); dh["v_feat"] = dh["v"]
STATE_FEATS = ["Segment_t", "Segment_t-1", "Segment_t-2", "StateDuration"]
FEATS = {"with K-Means states": CURRENT_FEATURES + ["v_feat"] + STATE_FEATS,      # = the model of notebook 06
         "without K-Means states": CURRENT_FEATURES + ["v_feat"]}
PARAMS = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss")
trp = dh[(dh.split == "train") & (dh.v > 0)]
y_tr = (trp.rev_next == 0).astype(int)
te = te.merge(dh.loc[dh.split == "test", ["CustomerID", "origin"] + [c for c in FEATS["with K-Means states"] if c != "Segment_t"]],
              on=["CustomerID", "origin"], how="left")
act = (te.v > 0).values
models, rows = {}, []
for name, F in FEATS.items():
    clf = XGBClassifier(**PARAMS).fit(trp[F].astype(float), y_tr)
    p = np.where(act, clf.predict_proba(te[F].astype(float))[:, 1], 0.0)
    col = "EL_stop" if name.startswith("with ") else "EL_stop_noseg"
    te["p_" + col] = p; te[col] = p * te.v
    rows.append({"model": name, "features": len(F), "AUC": roc_auc_score(te.loc[act, "stop"], p[act]),
                 "Brier": brier_score_loss(te.loc[act, "stop"], p[act]),
                 **{f"L_stop captured top {int(f*100)}%": captured_share(te.assign(L=te.L_stop), col, f) for f in (0.05, 0.10, 0.20)}})
ablation = pd.DataFrame(rows)
stop_spec = pd.Series({"target": "1 if the customer had revenue in the quarter ending at t and none in the next quarter",
    "training rows": f"{len(trp):,} (history-eligible train origins S3–S5 with revenue > 0)",
    "positive rate (train)": f"{y_tr.mean():.1%}", "test rows with revenue": f"{act.sum():,}",
    "positive rate (test)": f"{te.loc[act, 'stop'].mean():.1%}",
    "hyperparameters": "400 trees, depth 4, learning rate 0.05, subsample 0.8, colsample 0.8 (fixed, not tuned)"})
display(stop_spec.to_frame("value")); ablation

,value
target,1 if the customer had revenue in the quarter e...
training rows,"3,996 (history-eligible train origins S3–S5 wi..."
positive rate (train),37.2%
test rows with revenue,"3,125"
positive rate (test),34.4%
hyperparameters,"400 trees, depth 4, learning rate 0.05, subsam..."


,model,features,AUC,Brier,L_stop captured top 5%,L_stop captured top 10%,L_stop captured top 20%
0,with K-Means states,13,0.7626,0.1904,0.4199,0.6048,0.8284
1,without K-Means states,9,0.7568,0.1942,0.4162,0.6018,0.8320


In [3]:
tmp = te.assign(L=te.L_stop)
d = bootstrap_captured_diff(tmp, "EL_stop", "EL_stop_noseg", 0.10, n_boot=1000, seed=RANDOM_STATE)
pt = captured_share(tmp, "EL_stop", 0.10) - captured_share(tmp, "EL_stop_noseg", 0.10)
ablation_ci = pd.Series({"difference (with − without), top 10%": pt, "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5)})
# calibration of the stop-buying model (10 bins of predicted probability)
q = pd.qcut(te.loc[act, "p_EL_stop"], 10, duplicates="drop")
calib_stop = pd.DataFrame({"predicted": te.loc[act, "p_EL_stop"], "observed": te.loc[act, "stop"]}).groupby(q).mean()
display(ablation_ci.to_frame("value")); calib_stop

,value
"difference (with − without), top 10%",0.0031
ci_low,-0.0090
ci_high,0.0227


,predicted,observed
p_EL_stop,,
"(0.00152, 0.0279]",0.0141,0.0351
"(0.0279, 0.0771]",0.0500,0.0929
"(0.0771, 0.157]",0.1146,0.1534
"(0.157, 0.263]",0.2058,0.2276
"(0.263, 0.362]",0.3151,0.3131
"(0.362, 0.459]",0.4112,0.3878
"(0.459, 0.561]",0.5115,0.4872
"(0.561, 0.67]",0.6174,0.4633
"(0.67, 0.774]",0.7222,0.5962


## 2. Results by test origin (S6: full 3-month period; S7: the S7→S8 period of about 2.3 months)

In [4]:
by_origin = []
for o, g in te.groupby("origin"):
    gg = g.assign(L=g.L_stop)
    by_origin.append({"origin": o, "next period days": int(rev.loc[rev.snapshot == g.target_snapshot.iloc[0], "days"].iloc[0]),
                      "rows": len(g), "rows with revenue": int((g.v > 0).sum()), "stop rate": g.loc[g.v > 0, "stop"].mean(),
                      "L_stop total (£/30d)": g.L_stop.sum(),
                      **{f"captured {s}": captured_share(gg, s, 0.10, by="origin") for s in ["v", "SEG_EL", "EL_state", "EL_stop"]}})
by_origin = pd.DataFrame(by_origin)
by_origin["stop-model − value"] = by_origin["captured EL_stop"] - by_origin["captured v"]
by_origin

,origin,next period days,rows,rows with revenue,stop rate,L_stop total (£/30d),captured v,captured SEG_EL,captured EL_state,captured EL_stop,stop-model − value
0,S6,92,4303,1506,0.3466,"71,938.0385",0.4003,0.3457,0.3630,0.5308,0.1305
1,S7,70,4680,1619,0.3410,"94,503.6065",0.4666,0.4318,0.4818,0.6612,0.1946


## 3. Markov-order G-test for each training origin (no pooling over time)

Each origin S3, S4 and S5 is tested separately; within one origin each customer appears only once, so observations are independent across customers.

In [5]:
tr = ds[ds["history_eligible"] & (ds.split == "train")]
g_rows = []
for o, go in tr.groupby("origin"):
    G, dof = 0.0, 0
    for s in range(N_STATES):
        tab = pd.crosstab(go.loc[go.Segment_t == s, "Segment_t-1"], go.loc[go.Segment_t == s, TARGET])
        tab = tab.loc[tab.sum(axis=1) > 0, tab.sum(axis=0) > 0]
        if min(tab.shape) < 2: continue
        g, _, df_, _ = stats.chi2_contingency(tab, lambda_="log-likelihood"); G += g; dof += df_
    g_rows.append({"origin": o, "n": len(go), "G": G, "dof": dof, "p-value": stats.chi2.sf(G, dof)})
markov_by_origin = pd.DataFrame(g_rows)
markov_by_origin

,origin,n,G,dof,p-value
0,S3,2132,177.8319,24,0.0000
1,S4,2946,242.1215,22,0.0000
2,S5,3530,292.8093,21,0.0000


## 4. Tuning the stop-buying model on temporal folds

The same two expanding-window folds as Phase 4 (train S3 → validate S4; train S3–S4 → validate S5; states re-fitted inside each fold). The grid matches the Phase 4 XGBoost grid (16 points, without the class-weighting parameter). The selection criterion is fixed in advance: **mean AUC over the two folds**. The selected model is refitted on all training data and evaluated once on the test set.

In [6]:
import pickle, itertools
folds = pickle.load(open(INTERIM / "tuning_folds.pkl", "rb"))
GRID = [dict(zip(["n_estimators", "max_depth", "learning_rate", "min_child_weight"], v))
        for v in itertools.product([200, 500], [3, 5], [0.03, 0.1], [1, 5])]
F_STOP = FEATS["with K-Means states"]
def stop_rows(d):
    d = attach_revenue(d, rev); d = d[d.v > 0].copy(); d["v_feat"] = d["v"]; return d
fold_rows = {fid: (stop_rows(tr), stop_rows(va)) for fid, (tr, va) in folds.items()}
tune_rows = []
for g in GRID:
    aucs = []
    for fid, (tr_f, va_f) in fold_rows.items():
        m = XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss", **g)
        m.fit(tr_f[F_STOP].astype(float), (tr_f.rev_next == 0).astype(int))
        aucs.append(roc_auc_score((va_f.rev_next == 0).astype(int), m.predict_proba(va_f[F_STOP].astype(float))[:, 1]))
    tune_rows.append({**g, "fold1 AUC": aucs[0], "fold2 AUC": aucs[1], "mean AUC": np.mean(aucs)})
stop_tuning = pd.DataFrame(tune_rows).sort_values("mean AUC", ascending=False)
best_g = {k: (int(v) if k != "learning_rate" else float(v)) for k, v in stop_tuning.iloc[0][["n_estimators", "max_depth", "learning_rate", "min_child_weight"]].items()}
# fixed point used in notebook 06 (400 trees, depth 4, lr 0.05, mcw 1) — evaluated on the same folds for comparison
fixed_auc = []
for fid, (tr_f, va_f) in fold_rows.items():
    m = XGBClassifier(**PARAMS).fit(tr_f[F_STOP].astype(float), (tr_f.rev_next == 0).astype(int))
    fixed_auc.append(roc_auc_score((va_f.rev_next == 0).astype(int), m.predict_proba(va_f[F_STOP].astype(float))[:, 1]))
print("best grid point:", best_g, "| validation AUC best", round(stop_tuning['mean AUC'].iloc[0], 4), "| fixed setting", round(np.mean(fixed_auc), 4))
tuned = XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss", **best_g).fit(trp[F_STOP].astype(float), y_tr)
te["p_stop_tuned"] = np.where(act, tuned.predict_proba(te[F_STOP].astype(float))[:, 1], 0.0)
te["EL_stop_tuned"] = te.p_stop_tuned * te.v
tmp = te.assign(L=te.L_stop)
tuned_eval = pd.DataFrame([{"model": n, "validation mean AUC": va, "test AUC": roc_auc_score(te.loc[act, "stop"], te.loc[act, pc]),
                            "test Brier": brier_score_loss(te.loc[act, "stop"], te.loc[act, pc]),
                            "L_stop captured top 10%": captured_share(tmp, sc, 0.10)}
                           for n, va, pc, sc in [("fixed (notebook 06)", np.mean(fixed_auc), "p_EL_stop", "EL_stop"),
                                                 ("tuned", stop_tuning["mean AUC"].iloc[0], "p_stop_tuned", "EL_stop_tuned")]])
ci_t = {}
for a, b in [("EL_stop_tuned", "EL_stop"), ("EL_stop_tuned", "v")]:
    dd = bootstrap_captured_diff(tmp, a, b, 0.10, n_boot=1000, seed=RANDOM_STATE)
    ci_t[f"{a} − {b}"] = {"difference": captured_share(tmp, a, 0.10) - captured_share(tmp, b, 0.10), "ci_low": np.percentile(dd, 2.5), "ci_high": np.percentile(dd, 97.5)}
tuned_ci = pd.DataFrame(ci_t).T
display(stop_tuning.head(5)); display(tuned_eval); tuned_ci

best grid point: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 1} | validation AUC best 0.7638 | fixed setting 0.7416


,n_estimators,max_depth,learning_rate,min_child_weight,fold1 AUC,fold2 AUC,mean AUC
0,200,3,0.0300,1,0.7588,0.7688,0.7638
1,200,3,0.0300,5,0.7559,0.7699,0.7629
4,200,5,0.0300,1,0.7485,0.7720,0.7602
5,200,5,0.0300,5,0.7470,0.7713,0.7592
8,500,3,0.0300,1,0.7458,0.7657,0.7557


,model,validation mean AUC,test AUC,test Brier,L_stop captured top 10%
0,fixed (notebook 06),0.7416,0.7626,0.1904,0.6048
1,tuned,0.7638,0.7766,0.1793,0.5959


,difference,ci_low,ci_high
EL_stop_tuned − EL_stop,-0.0090,-0.0238,0.0089
EL_stop_tuned − v,0.1579,0.1176,0.2027


## 5. Configuration A on all customers (including new customers)

Phase 4 uses only customers already observed at t − 2 so that A and B are compared fairly. Here Configuration A (which needs no history) is trained and tested on **all** customers at the same origins (train S3–S5, test S6–S7), with the parameters selected in Phase 4. Purpose: to see how excluding new customers affects the Recent Low-Frequency class and the comparison with persistence.

In [7]:
import ast
import modeling as M, bootstrap as BS
full = pd.read_parquet(PROCESSED / "prediction_dataset.parquet")
tr_all = full[(full.split == "train") & full.origin.isin(["S3", "S4", "S5"])]
te_all = full[full.split == "test"]
bp = pd.read_csv(TABLES / "phase4_best_params.csv")
y_all = te_all[TARGET].astype(int).values; seg_all = te_all["Segment_t"].astype(int).values
preds_all = {"Persistence": seg_all}
for m in ["LogReg", "RandomForest", "XGBoost"]:
    prm = ast.literal_eval(bp[(bp.model == m) & (bp.config == "A")]["params"].iloc[0])
    pipe = M.fit(m, M.make_model(m, M.CONFIGS["A"], prm), *M.to_xy(tr_all, M.CONFIGS["A"]))
    preds_all[m] = pipe.predict(M.to_xy(te_all, M.CONFIGS["A"])[0])
cust = te_all.CustomerID.values; uniq = np.unique(cust); W = BS.multiplicities(len(uniq), 2000, RANDOM_STATE)
def cmf(pred):
    u, Cm = BS.customer_confusions(cust, y_all, pred, N_STATES); Cf = np.zeros((len(uniq), N_STATES ** 2)); Cf[np.searchsorted(uniq, u)] = Cm; return Cf
Cp = cmf(preds_all["Persistence"]); dp = BS.cm_metrics(W @ Cp, N_STATES)["macro_f1"]
rows = []
for m, pr in preds_all.items():
    pc = M.per_class(y_all, pr); Cm = cmf(pr); dm = BS.cm_metrics(W @ Cm, N_STATES)["macro_f1"]
    est = M.f1_score(y_all, pr, average="macro", labels=M.LABELS) - M.f1_score(y_all, preds_all["Persistence"], average="macro", labels=M.LABELS)
    rows.append({"model": m, "Macro-F1": M.f1_score(y_all, pr, average="macro", labels=M.LABELS), "F1 Recent": pc.loc[2, "f1"],
                 "accuracy": M.accuracy_score(y_all, pr), "changers correct": M.accuracy_score(y_all[y_all != seg_all], pr[y_all != seg_all]),
                 "− persistence": est, "ci_low": np.percentile(dm - dp, 2.5) if m != "Persistence" else np.nan,
                 "ci_high": np.percentile(dm - dp, 97.5) if m != "Persistence" else np.nan})
all_customers = pd.DataFrame(rows)
info = pd.Series({"train rows (all customers)": len(tr_all), "train rows (history-eligible, Phase 4)": int(tr_all.history_eligible.sum()),
                  "test rows (all customers)": len(te_all), "test rows (history-eligible)": int(te_all.history_eligible.sum()),
                  "Recent share of targets, all test rows": (y_all == 2).mean(),
                  "Recent share of targets, history-eligible": (te_all.loc[te_all.history_eligible, TARGET].astype(int) == 2).mean()})
display(info.to_frame("value")); all_customers

,value
train rows (all customers),"12,513.0000"
"train rows (history-eligible, Phase 4)","8,608.0000"
test rows (all customers),"10,408.0000"
test rows (history-eligible),"8,983.0000"
"Recent share of targets, all test rows",0.0599
"Recent share of targets, history-eligible",0.0431


,model,Macro-F1,F1 Recent,accuracy,changers correct,− persistence,ci_low,ci_high
0,Persistence,0.6639,0.2146,0.7778,0.0000,0.0000,NaN,NaN
1,LogReg,0.6222,0.0000,0.8118,0.2317,-0.0417,-0.0498,-0.0336
2,RandomForest,0.6733,0.1857,0.8117,0.2369,0.0094,0.0014,0.0174
3,XGBoost,0.6593,0.1699,0.7893,0.2901,-0.0046,-0.0148,0.0054


In [8]:
stop_spec.to_frame("value").to_csv(TABLES / "phase7_stop_model_spec.csv")
ablation.to_csv(TABLES / "phase7_kmeans_ablation.csv", index=False)
ablation_ci.to_frame("value").to_csv(TABLES / "phase7_kmeans_ablation_ci.csv")
calib_stop.to_csv(TABLES / "phase7_stop_model_calibration.csv")
by_origin.to_csv(TABLES / "phase7_by_origin.csv", index=False)
markov_by_origin.to_csv(TABLES / "phase7_markov_by_origin.csv", index=False)
print("saved")
stop_tuning.to_csv(TABLES / "phase7_stop_model_tuning.csv", index=False)
tuned_eval.to_csv(TABLES / "phase7_stop_model_tuned_eval.csv", index=False)
tuned_ci.to_csv(TABLES / "phase7_stop_model_tuned_ci.csv")
info.to_frame("value").to_csv(TABLES / "phase7_all_customers_info.csv")
all_customers.to_csv(TABLES / "phase7_all_customers_configA.csv", index=False)
print("saved all")

saved
saved all
